In [ ]:
import os
import gc
import subprocess
import pickle
from multiprocessing import Pool

import numpy as np
import pandas as pd
import tifffile
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.spatial import cKDTree, KDTree
from numba import njit
import seaborn as sns

import xgboost as xgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer

import cell_detection as cd

In [ ]:
def makemask(size=11):
    """Create a 3D Manhattan-distance mask."""
    center = size // 2
    mask = np.zeros((size, size, size), dtype=int)
    for x in range(size):
        for y in range(size):
            for z in range(size):
                if abs(x - center) + abs(y - center) + abs(z - center) <= center:
                    mask[x, y, z] = 1
    return mask.astype(np.float64)


def resize3D(img, original_size, after_size):
    """Resize a 3D volume by applying 2D cv2.resize twice (z-first, then y-x)."""
    ratio = original_size / after_size
    img = img.astype(np.float32)

    tmp = [cv2.resize(sl, dsize=None, fx=ratio, fy=ratio) for sl in img]
    tmp = np.array(tmp).T  # swap axes

    tmp2 = [cv2.resize(sl, dsize=None, fx=ratio, fy=1) for sl in tmp]
    tmp2 = np.array(tmp2)
    return tmp2.T


def min_max(x: np.ndarray):
    """Min-max normalize to [0, 1] with keepdims."""
    minval = x.min(axis=None, keepdims=True)
    maxval = x.max(axis=None, keepdims=True)
    result = (x - minval) / (maxval - minval + 1e-12)
    return result


def makePSF(size, sigma_values):
    """Generate PSF bank by Gaussian-blurring an impulse and shifting (3x3x3)."""
    all_psfs = []
    target = np.zeros((size * 2 + 1, size * 2 + 1, size * 2 + 1))
    center = (target.shape[0] // 2, target.shape[1] // 2, target.shape[2] // 2)
    target[center] = 1

    for sigma in sigma_values:
        base = ndi.gaussian_filter(target, sigma=sigma)
        psflist = []
        for i in range(-1, 2):
            for j in range(-1, 2):
                for k in range(-1, 2):
                    t = np.roll(np.roll(np.roll(base, i, axis=0), j, axis=1), k, axis=2)
                    resized = resize3D(t, size, size * 2 + 1)
                    psflist.append(resized)
        psflist = [min_max(p) for p in psflist]
        all_psfs.extend(psflist)

    return np.array(all_psfs)


def read_tiff_stack(folder_path, start_index=0, num_images=-1):
    """Load a TIFF stack from a folder containing sequential TIFF files."""
    tiff_files = [
        os.path.join(folder_path, fn)
        for fn in os.listdir(folder_path)
        if fn.lower().endswith(('.tif', '.tiff'))
    ]
    tiff_files.sort()

    if num_images == -1:
        tiff_files = tiff_files[start_index:]
    else:
        tiff_files = tiff_files[start_index:(start_index + num_images)]

    print(f"Load tiff: {start_index}-{len(tiff_files)}")

    first = tifffile.imread(tiff_files[0])
    stack = np.empty((len(tiff_files),) + first.shape, dtype=np.uint16)

    for i, tf in enumerate(tiff_files):
        stack[i] = tifffile.imread(tf)

    return stack


class UnionFind:
    """Disjoint-set union (union-find) structure."""
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [1] * n

    def find(self, u):
        if u != self.parent[u]:
            self.parent[u] = self.find(self.parent[u])
        return self.parent[u]

    def union(self, u, v):
        ru, rv = self.find(u), self.find(v)
        if ru != rv:
            if self.rank[ru] > self.rank[rv]:
                self.parent[rv] = ru
            elif self.rank[ru] < self.rank[rv]:
                self.parent[ru] = rv
            else:
                self.parent[rv] = ru
                self.rank[ru] += 1


def merge_lists(lists):
    """
    Merge integer lists by connectivity: any numbers appearing in the same list
    are considered connected and will be grouped together.
    """
    max_value = max(max(lst) for lst in lists) + 1
    uf = UnionFind(max_value)

    # Union within each list
    for lst in lists:
        first = lst[0]
        for num in lst[1:]:
            uf.union(first, num)

    # Collect groups
    groups = {}
    for lst in lists:
        root = uf.find(lst[0])
        if root not in groups:
            groups[root] = set()
        for num in lst:
            groups[root].add(num)

    return [sorted(list(group)) for group in groups.values()]


@njit
def normalize_image(img):
    """Min-max normalize to [0, 1]."""
    img_min = img.min()
    img_max = img.max()
    return (img - img_min) / (img_max - img_min + 1e-12)


@njit
def calculate_centroid_and_inertia_tensor_center_based(img):
    """Compute centroid and inertia tensor around the volume center."""
    norm_img = normalize_image(img)
    shape = norm_img.shape

    z, y, x = np.indices(shape)
    total_mass = np.sum(norm_img) + 1e-12

    # Centroid (intensity-weighted)
    centroid_x = np.sum(x * norm_img) / total_mass
    centroid_y = np.sum(y * norm_img) / total_mass
    centroid_z = np.sum(z * norm_img) / total_mass
    centroid = np.array([centroid_x, centroid_y, centroid_z])

    # Volume center
    center_x = shape[2] / 2
    center_y = shape[1] / 2
    center_z = shape[0] / 2
    center = np.array([center_x, center_y, center_z])

    # Shifted coordinates
    x_shifted = x - center_x
    y_shifted = y - center_y
    z_shifted = z - center_z

    # Inertia tensor
    Ixx = np.sum(norm_img * (y_shifted**2 + z_shifted**2))
    Iyy = np.sum(norm_img * (x_shifted**2 + z_shifted**2))
    Izz = np.sum(norm_img * (x_shifted**2 + y_shifted**2))
    Ixy = -np.sum(norm_img * x_shifted * y_shifted)
    Ixz = -np.sum(norm_img * x_shifted * z_shifted)
    Iyz = -np.sum(norm_img * y_shifted * z_shifted)

    inertia_tensor = np.array([[Ixx, Ixy, Ixz],
                               [Ixy, Iyy, Iyz],
                               [Ixz, Iyz, Izz]])

    # Eccentricity (centroid offset from center)
    eccentricity = centroid - center

    return center, centroid, eccentricity, inertia_tensor


def create_dataframe(d1p, d1n):
    """Create a DataFrame for 'pos' and 'neg' groups from feature arrays."""
    # Create DataFrame for the 'pos' group
    df_pos = pd.DataFrame({
        'Group': ['pos'] * len(d1p),
        'Hellinger Coefficient': [dp[0] for dp in d1p],
        'Chernoff Coefficient': [dp[1] for dp in d1p],
        'Jeffreys Distance': [dp[2] for dp in d1p],
        'Directed Divergence': [dp[3] for dp in d1p],
        'J-Divergence': [dp[4] for dp in d1p],
        'L1': [dp[5] for dp in d1p],
        'L2': [dp[6] for dp in d1p],
        'Kullback-Leibler': [dp[7] for dp in d1p],
        'Pearson': [dp[8] for dp in d1p],
        'Jensen-Shannon': [dp[9] for dp in d1p],
        'ratio I': [dp[10] for dp in d1p],
        'delta I': [dp[11] for dp in d1p],
        'exp': [dp[12] for dp in d1p],
        'CT': [dp[13] for dp in d1p],
        'region': [dp[14] for dp in d1p],
        'X': [dp[15] for dp in d1p],
        'Y': [dp[16] for dp in d1p],
        'Z': [dp[17] for dp in d1p]
    })

    # Create DataFrame for the 'neg' group
    df_neg = pd.DataFrame({
        'Group': ['neg'] * len(d1n),
        'Hellinger Coefficient': [dn[0] for dn in d1n],
        'Chernoff Coefficient': [dn[1] for dn in d1n],
        'Jeffreys Distance': [dn[2] for dn in d1n],
        'Directed Divergence': [dn[3] for dn in d1n],
        'J-Divergence': [dn[4] for dn in d1n],
        'L1': [dn[5] for dn in d1n],
        'L2': [dn[6] for dn in d1n],
        'Kullback-Leibler': [dn[7] for dn in d1n],
        'Pearson': [dn[8] for dn in d1n],
        'Jensen-Shannon': [dn[9] for dn in d1n],
        'ratio I': [dn[10] for dn in d1n],
        'delta I': [dn[11] for dn in d1n],
        'exp': [dn[12] for dn in d1n],
        'CT': [dn[13] for dn in d1n],
        'region': [dn[14] for dn in d1n],
        'X': [dn[15] for dn in d1n],
        'Y': [dn[16] for dn in d1n],
        'Z': [dn[17] for dn in d1n]
    })
    
    print(len(df_pos))
    print(len(df_neg))

    # Concatenate and return
    df = pd.concat([df_pos, df_neg], ignore_index=True)
    return df


@njit
def GetDistance(data, psfs, mask):
    """Calculate multiple distance metrics between data and PSFs within the given mask."""
    z, y, x = np.shape(data)
    results = [np.inf] * 10  # Initialize 10 distance metrics

    for psf in psfs:
        distances = [0.0] * 10
        for i in range(z):
            for j in range(y):
                for k in range(x):
                    if mask[i, j, k] > 0:
                        if data[i, j, k] > 0 and psf[i, j, k] > 0:
                            m_ratio = data[i, j, k] / psf[i, j, k]
                            s_ratio = data[i, j, k] + psf[i, j, k]

                            # Positive-value metrics
                            distances[0] += (data[i, j, k] * psf[i, j, k]) ** 0.5  # Hellinger
                            distances[1] += data[i, j, k] ** data[i, j, k] * psf[i, j, k] ** (1.0 - data[i, j, k])  # Chernoff
                            distances[3] += data[i, j, k] * np.log2(m_ratio)  # Directed divergence
                            distances[4] += (data[i, j, k] - psf[i, j, k]) * np.log2(m_ratio)  # J-divergence
                            distances[7] += ((data[i, j, k] - psf[i, j, k]) ** 3) / (mask[i, j, k] ** 3)  # KM divergence

                            if s_ratio > 0:
                                distances[9] += (
                                    data[i, j, k] * np.log10(2.0 * data[i, j, k] / s_ratio)
                                    + psf[i, j, k] * np.log10(2.0 * psf[i, j, k] / s_ratio)
                                )  # JS divergence

                        # Metrics calculated regardless of positivity
                        distances[2] += (data[i, j, k] ** 0.5 - psf[i, j, k] ** 0.5) ** 2.0  # Jeffreys
                        distances[5] += abs(data[i, j, k] - psf[i, j, k])  # L1 norm
                        distances[6] += (data[i, j, k] - psf[i, j, k]) ** 2  # L2 norm squared
                        distances[8] += (
                            ((data[i, j, k] - psf[i, j, k]) ** 2) / data[i, j, k] if data[i, j, k] != 0 else 0
                        )  # Pearson

        distances[6] = distances[6] ** 0.5  # Convert L2^2 to L2
        results = [min(r, d) for r, d in zip(results, distances)]

    return results


def Machine_Learning(df, features, test_size=0.3, random_state=42):
    """Train and evaluate multiple ML models using the specified features."""
    data = df[features]
    TF = df['Group'].map({'pos': 1, 'neg': 0})

    # Split data
    data_train, data_test, TF_train, TF_test = train_test_split(
        data, TF, test_size=test_size, random_state=random_state
    )

    # Initialize models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Support Vector Machine (SVM)": SVC(),
        "k-Nearest Neighbors (KNN)": KNeighborsClassifier(),
        "Gradient Boosting": GradientBoostingClassifier(),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "Decision Tree": DecisionTreeClassifier(),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(max_iter=1000),
        "Naive Bayes": GaussianNB(),
        "eXtreme Gradient Boosting (XGBoost)": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
        "LightGBM": LGBMClassifier(),
        "Adaptive Boosting (AdaBoost)": AdaBoostClassifier()
    }

    # Train and evaluate
    results = {}
    for name, model in models.items():
        model.fit(data_train, TF_train)
        pred = model.predict(data_test)
        accuracy = accuracy_score(TF_test, pred)
        report = classification_report(TF_test, pred)
        f1 = f1_score(TF_test, pred, average='weighted')
        results[name] = (accuracy, report, f1, model)

        print("\n******************************************************************")
        print(f"Model: {name}")
        print(f"Accuracy: {accuracy}")
        print(report)
        print(f"F1 Score: {f1}")

        # Feature importance (if available)
        if name == "Logistic Regression":
            print("Coefficients:", model.coef_)
            print("Intercept:", model.intercept_)
            feature_importances = pd.DataFrame(model.coef_.flatten(),
                                               index=data_train.columns,
                                               columns=['importance']).sort_values('importance', ascending=False)
            print(feature_importances)
        elif name in ["Gradient Boosting", "Random Forest", "XGBoost", "AdaBoost", "Decision Tree"]:
            feature_importances = pd.DataFrame(model.feature_importances_,
                                               index=data_train.columns,
                                               columns=['importance']).sort_values('importance', ascending=False)
            print(feature_importances)
        else:
            print("Feature importances are not available for this model.")

    # Plot F1 scores
    f1_scores = [(name, vals[2]) for name, vals in results.items()]
    f1_scores.sort(key=lambda x: x[1], reverse=True)
    names, scores = zip(*f1_scores)

    plt.figure(figsize=(12, 8))
    plt.bar(names, scores, color='skyblue')
    plt.axhline(y=0.9, color='r', linestyle='--', label='Threshold (0.9)')
    plt.xlabel('Model')
    plt.ylabel('F1 Score')
    plt.ylim(0, 1.05)
    plt.title('Comparison of F1 Scores Across Different Models')
    plt.xticks(rotation=45)
    plt.legend()
    plt.show()

    return results


def Learn_save_best(df, features, model_one, model_name, test_size=0.3, random_state=42):
    """Fit and save a chosen model on a train/test split."""
    data = df[features]
    TF = df["Group"].map({"pos": 1, "neg": 0})
    
    X_tr, X_te, y_tr, y_te = train_test_split(data, TF, test_size=test_size, random_state=random_state)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Support Vector Machine (SVM)": SVC(),
        "k-Nearest Neighbors (KNN)": KNeighborsClassifier(),
        "Gradient Boosting": GradientBoostingClassifier(),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "Decision Tree": DecisionTreeClassifier(),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(max_iter=1000),
        "Naive Bayes": GaussianNB(),
        "eXtreme Gradient Boosting (XGBoost)": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
        "LightGBM": LGBMClassifier(),
        "Adaptive Boosting (AdaBoost)": AdaBoostClassifier(),
    }

    model = models[model_one]
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    acc = accuracy_score(y_te, pred)
    rep = classification_report(y_te, pred)
    f1w = f1_score(y_te, pred, average="weighted")

    X_te = X_te.copy()
    X_te["predict"] = pred
    print("save trained model")

    # Save model
    out_dir = os.path.join(src, "cd_model")
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, model_name + ".pkl"), "wb") as f:
        pickle.dump(model, f)

    return X_te


def display_images_grid(images, grid_size=(5, 5), figsize=(10, 10)):
    """Display a grid of example images."""
    fig, axes = plt.subplots(grid_size[0], grid_size[1], figsize=figsize)
    for i, ax in enumerate(axes.flat):
        if i < len(images):
            ax.imshow(images[i][5])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def plot_swarm(dataframe, features):
    """Display swarm plots for each metric."""
    plt.figure(figsize=(15, 10))

    for i, metric in enumerate(features, 1):
        if metric not in dataframe.columns:
            continue
        plt.subplot(4, 3, i)
        sns.stripplot(data=dataframe, x='Group', y=metric, jitter=True, size=3)
        plt.title(metric)

    plt.tight_layout()
    plt.show()


def Make2Dplots(df, features):
    """Generate pairplots of features with group coloring."""
    plot_data = df[features + ['Group']]
    sns.pairplot(plot_data, hue='Group', markers=["o", "s"],
                 palette='bright', diag_kind='kde', plot_kws={'s': 3})
    plt.suptitle('Pairplot of Features', size=30)
    plt.subplots_adjust(top=0.95)
    plt.show()

In [ ]:
# Calculate distances between candidate ROIs and PSFs using multiprocessing
def multi_calc(df, imgdir, pkldir, savedir, mask_size, cores, overlap, z_num, slice_num, sl_block, outf, size, blockdim_x, blockdim_y):
    args = [] 
    for i in range(sl_block):
        offset = i * slice_num
        if i == 0:
            length = slice_num + overlap
            st = offset
        else:
            length = slice_num + 2 * overlap
            st = offset - overlap

        if st + length > z_num:
            print("z_num over end")
            break

        bin_path = os.path.join(savedir, f"{outf}{i}.bin")
        if not os.path.exists(bin_path):
            args.append((df, imgdir, pkldir, savedir, mask_size, i, st, length, offset, slice_num, outf, size, blockdim_x, blockdim_y))
            
    with Pool(processes=cores) as pool:
        pool.map(calc_dist, args)


def normalize(image: np.ndarray):
    """Min-max normalize to [0, 1]."""
    return (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-12)


def calc_dist(args):
    """Extract ROIs from TIFF stack and compute distances."""
    df, imgdir, pkldir, savedir, mask_size, i, st, length, offset, slice_num, outf, size, blockdim_x, blockdim_y = args
    print(i)

    center = size // 2

    # Select subset of data within Z range
    df_c = df[(df["Z"] >= offset) & (df["Z"] < offset + slice_num)]

    # Load TIFF stack
    img = read_tiff_stack(imgdir, start_index=st, num_images=length)

    # Prepare output array
    point_np = np.zeros((len(df_c), mask_size), dtype="float32")

    # Save coordinates
    df_c.to_pickle(os.path.join(pkldir, f"coordinates{i}.pkl"))

    # Extract ROI for each coordinate
    for m in range(len(df_c)):
        x = int(df_c.iloc[m]["X"] - center)
        y = int(df_c.iloc[m]["Y"] - center)
        z = int(df_c.iloc[m]["Z"] - center)
        roi = img[int(z - st):int(z - st + size),
                  int(y):int(y + size),
                  int(x):int(x + size)]

        if roi.shape == mask_size:
            point_np[m] = np.ravel(roi)

    del img
    gc.collect()

    # Save raw ROI data
    np.save(os.path.join(savedir, f"roi{i}.npy"), point_np.T)

    # Normalize and save again
    for m in range(len(point_np)):
        point_np[m] = normalize(point_np[m])
    np.save(os.path.join(savedir, f"roi_norm{i}.npy"), point_np.T)

    del point_np
    gc.collect()

    # Execute external distance calculation
    args_str = f"{savedir} {outf} psf mask {i} {size} {blockdim_x} {blockdim_y}"
    cmd = f"./cal_d {args_str}"
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"

acc_dirs = ["circadian_1st", "circadian_2nd"]
deconv_fs = ["circadian_1st/circadian_1st_Deconv", "circadian_2nd/circadian_2nd_Deconv"]

In [ ]:
# Generate and save PSF bank and mask

size = 11  # ROI/mask size (voxels)
psf_f = "psf"
mask_f = "mask"

# Create mask and PSFs
mask = makemask(size)
psfs = makePSF(size, psf_sigmas)
psfs = [normalize(p.astype(np.float64)) for p in psfs]

# Flatten PSFs into 2D array: (n_psfs, size³)
psfs_np = np.array([p.ravel() for p in psfs], dtype=np.float64)

# Save PSFs and mask
np.save(os.path.join(dst, "cfos_roi", psf_f), psfs_np.T)
np.save(os.path.join(dst, "cfos_roi", mask_f), mask.ravel().astype(np.int32))

In [ ]:
fpr = 0.5  # threshold for false positive rate
dist_num = 10      # number of features
slice_num = 200
overlap = 5
outf = "dist"
blockdim_y = len(psfs)  # GPU block y (number of PSFs)
blockdim_x = 1          # GPU block x
cores = 12              # CPU cores

postfix = f"output_peak_ratioI_fpr{fpr}"
postfix_all = "output_peak"

FPw = os.path.join(dst, "cfos_teachers", f"deconv_detection_ratioI_fpr{fpr}")
os.makedirs(FPw, exist_ok=True)
exps = ["1st", "2nd"]

features = [
    'Hellinger Coefficient', 'Chernoff Coefficient', 'Jeffreys Distance',
    'Directed Divergence', 'J-Divergence', 'L1', 'L2', 'Kullback-Leibler',
    'Pearson', 'Jensen-Shannon', 'ratio I', 'delta I',
    'Tensor_xx', 'Tensor_yy', 'Tensor_zz'
]

psf_sigmas = [(3, 3, 3), (4, 4, 4), (4.5, 4.5, 4.5)]

In [ ]:
# Visualize generated PSFs (for verification)

display_images_grid(psfs)

In [ ]:
# Collect unique sample names

CT_li = np.arange(0, 48, 4)  # circadian time points (CT0–44, every 4 h)
sample_ids = np.arange(1, 7, 1)

reconsts = os.listdir(cfos_dir)
sample_names = []

for CT in CT_li:
    for sample_id in sample_ids:
        sample = f"CT{CT}_{str(sample_id).zfill(2)}"
        for reconst in reconsts:
            if sample in reconst:
                sample_names.append(sample)

print(len(sample_names))

In [ ]:
# Detect cell candidates by intensity peaks

structure_name = "SYTOX-G"   # Input directory name for structure image
color_name1 = "cfos"          # Input directory name for the 1st color
resolution = int(10 / 2.5)    # Input voxel size (μm)

# params = [intensity_threshold, deltaI_threshold, psf_threshold, line_threshold]
params = [5000, 0, None, None]

peak_size = 7
psf_size = 11
batchsize = 100
overlap = 10
threads = 20

cellnumber = []
# skip=1

for deconv_f in deconv_fs:
    FP = os.path.join(src, deconv_f)                          # Parent input dir
    FPout = os.path.join(dst, f"{deconv_f}_{postfix}")        # Parent output dir

    # List child dirs under current FP
    dirs = [d for d in os.listdir(FP) if os.path.isdir(os.path.join(FP, d))]

    for d in dirs:
        coord_csv = os.path.join(FPout, d, "coordinates.csv")
        if os.path.exists(coord_csv):
            print(f"{coord_csv} already exist")
            continue

        print(os.path.join(FP, d))

        # sample_name = "CT0_01"
        # if(sample_name in d):
        #     skip=0
        # if(skip==1):
        #     continue

        if color_name1 in d:
            FPr = os.path.join(FP, d)
            FPw = os.path.join(FPout, d)
            df = cd.GetPeaks(FPr, FPw, params, peak_size, psf_size, batchsize, overlap, threads, save_points=False)
            cellnumber.append((d, str(len(df))))
        # skip=1

In [ ]:
# Treat saturated peaks

moving_points_paths = [os.path.join(dst, d + "_" + postfix_all) for d in deconv_fs]

for l, exp in enumerate(exps):
    if l < 1:
        continue

    moving_points_path = moving_points_paths[l]
    moving_points_csv_li = os.listdir(moving_points_path)

    for i, sample in enumerate(sample_names):
        # find folder name that contains the sample (ignoring 'cfos_' prefix)
        for f in moving_points_csv_li:
            if sample in f.replace("cfos_", ""):
                break
        print(f)

        # skip if already processed
        if os.path.exists(os.path.join(moving_points_path, f, "coordinates2.csv")):
            print(exp, sample, "coordinates2.csv already exists")
            continue

        # read coordinates.csv (or coordinates_id.csv if present)
        coord_id_path = os.path.join(moving_points_path, f, "coordinates_id.csv")
        coord_path    = os.path.join(moving_points_path, f, "coordinates.csv")
        if os.path.exists(coord_id_path):
            df = pd.read_csv(coord_id_path)  # x>z
        else:
            if not os.path.exists(coord_path):
                print(exp, sample, "coordinates.csv not found")
                continue
            df = pd.read_csv(coord_path, engine='python')

        print("pre_df_len:", len(df))

        # saturated points
        df_i = df[df["intensity"] == 65535]
        print("saturated:", len(df_i))

        coords = df_i[['X', 'Y', 'Z']].values
        tree = cKDTree(coords)

        # group adjacent points within radius 1
        adjacent_points = [tree.query_ball_point(coords[idx], r=1) for idx in range(len(df_i))]

        merged_lists = merge_lists(adjacent_points)
        print(len(merged_lists))

        center_near = []
        for points in merged_lists:
            df_c = df_i.iloc[points]
            center = (np.sum(df_c["X"]) / len(df_c),
                      np.sum(df_c["Y"]) / len(df_c),
                      np.sum(df_c["Z"]) / len(df_c))

            coords_c = df_c[['X', 'Y', 'Z']].values
            tree_c = cKDTree(coords_c)
            distance, index = tree_c.query(np.array(center))

            # nearest point to the cluster center
            nearest = df_c.iloc[index]
            center_near.append(nearest)

        df_uni = pd.concat(center_near, axis=1).T.reset_index(drop=True)

        # keep non-saturated + representative center-near points
        df = df[df["intensity"] != 65535]
        df = pd.concat([df, df_uni], axis=0)
        print("new dataframe", len(df))

        out_csv = os.path.join(moving_points_path, f, "coordinates2.csv")
        df.to_csv(out_csv, index=False)

In [ ]:
# Obtain ratio I threshold by sum of peaks

fpr = 0.5

diff_log = []
for l, deconv_f in enumerate(deconv_fs):
    print(deconv_f)

    FPout     = os.path.join(dst, deconv_f + "_" + postfix)       # output parent
    FPout_pre = os.path.join(dst, deconv_f + "_" + postfix_all)   # input parent

    # list subdirectories
    dirs = [d for d in os.listdir(FPout_pre) if os.path.isdir(os.path.join(FPout_pre, d))]

    for j, dir in enumerate(dirs):
        print(dir)

        coord2 = os.path.join(FPout_pre, dir, "coordinates2.csv")
        if os.path.exists(coord2):
            df = pd.read_csv(coord2)
            print("all_peaks: ", dir, len(df))

            df["min"] = df["intensity"] - df["deltaI"]
            df["ratioI"] = df["intensity"] / df["min"]

            df.replace(np.inf, 65535, inplace=True)
            diff_log.append(np.log10(df["ratioI"]))

# stack to 1D array
for i, v in enumerate(diff_log):
    if i == 0:
        diff_log_all = np.array(v.tolist())
    else:
        diff_log_all = np.append(diff_log_all, np.array(v.tolist()))

# persist and GC
diff_log_all = diff_log_all.reshape(-1, 1)
np.save(os.path.join(dst, "log_ratioI_all"), diff_log_all)

# subsample (every 10th) after shuffling
np.random.shuffle(diff_log_all)
diff_log_all = diff_log_all[::10]

plt.hist(diff_log_all)

# fit Gaussian mixture model
initial_means = np.array([[0.09], [0.19], [0.5]])
gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',
    means_init=initial_means
).fit(diff_log_all)

sorted_indices = np.argsort(gmm.means_.flatten())

x = np.linspace(0, 1, 200)
gd1 = norm.pdf(x,
               gmm.means_[sorted_indices[0], -1],
               np.sqrt(gmm.covariances_[sorted_indices[0]][-1, -1]))
gd2 = norm.pdf(x,
               gmm.means_[sorted_indices[1], -1],
               np.sqrt(gmm.covariances_[sorted_indices[1]][-1, -1]))
# gd3 is unused in the original code

plt.figure(figsize=(10, 8))
plt.plot(x, gmm.weights_[sorted_indices[0]] * gd1, label='gd1', color="magenta")
plt.plot(x, gmm.weights_[sorted_indices[1]] * gd2, label='gd2', color="orange")
plt.plot(x,
         gmm.weights_[sorted_indices[0]] * gd1 + gmm.weights_[sorted_indices[1]] * gd2,
         label='gd1+gd2', color="brown")

plt.hist(diff_log_all, bins=300, density=True, alpha=0.4, label="data", color="skyblue")

boundary_gd1 = norm.ppf(1 - fpr / 100,
                        gmm.means_[sorted_indices[0], -1],
                        np.sqrt(gmm.covariances_[sorted_indices[0]][-1, -1]))

plt.axvline(boundary_gd1, color='red', linestyle='dashed', linewidth=2,
            label='99.5th Percentile Boundary')
plt.xlim(0, 1)

th = 10 ** boundary_gd1
print("ratioI threshold:", th)
plt.show()

np.savetxt(os.path.join(dst, f"ratioI_fpr{fpr}.txt"), np.array([th]))

In [ ]:
# Get dataframes cut by ratioI threshold

for l, deconv_f in enumerate(deconv_fs):
    print(deconv_f)

    FP         = os.path.join(dst, deconv_f)
    FPout      = os.path.join(dst, deconv_f + "_" + postfix)        # output parent
    FPout_pre  = os.path.join(dst, deconv_f + "_" + postfix_all)    # input parent

    dirs = [d for d in os.listdir(FP) if os.path.isdir(os.path.join(FP, d))]

    for j, dir in enumerate(dirs):
        print(dir)
        savedir = os.path.join(FPout, dir)
        os.makedirs(savedir, exist_ok=True)

        pkl_ratio_path = os.path.join(FPout_pre, dir, "coordinates2_ratioI.pkl")
        if not os.path.exists(pkl_ratio_path):
            print("ratioI pkl not found:", pkl_ratio_path)
            continue

        df = pd.read_pickle(pkl_ratio_path)
        print("all_peaks:", dir, len(df))

        # threshold by ratioI
        df = df[df["ratioI"] > th]
        print(f"num of peaks_in_{fpr}% False positive:", len(df))

        out_pkl = os.path.join(savedir, "coordinates.pkl")
        df.to_pickle(out_pkl)
        print("saved:", out_pkl)

In [ ]:
# Image ROI -> compute distance

for l, deconv_f in enumerate(deconv_fs):
    print(deconv_f)

    FP = os.path.join(src, deconv_f)
    FPout_pre = os.path.join(dst, f"{deconv_f}_{postfix_all}")
    FPout = os.path.join(dst, f"{deconv_f}_{postfix}")

    # Collect subdirectories
    dirs = [d for d in os.listdir(FPout_pre) if os.path.isdir(os.path.join(FPout_pre, d))]

    for j, dir in enumerate(dirs):
        savedir = os.path.join(dst, "cfos_roi", postfix, exps[l], dir)
        print(savedir)
        os.makedirs(savedir, exist_ok=True)

        imgdir = os.path.join(src, deconv_f, dir)
        pkldir = os.path.join(FPout, dir, "reconst")
        os.makedirs(pkldir, exist_ok=True)

        coord_pkl = os.path.join(FPout, dir, "coordinates.pkl")
        if os.path.exists(coord_pkl):
            df = pd.read_pickle(coord_pkl)
            df = df.sort_values("Z")
            print(df.iloc[0:3, :])

            z_dir = os.path.join(src, deconv_f, dir)
            z_num = len(os.listdir(z_dir))
            print("znum", z_num)

            sl_block = z_num // slice_num
            re = z_num % slice_num
            offset = 0

            if __name__ == "__main__":
                mask_size = mask.size
                multi_calc(df, imgdir, pkldir, savedir, mask_size, cores, overlap, z_num, slice_num, sl_block, outf, size, blockdim_x, blockdim_y)

            if re != 0:
                offset = sl_block * slice_num
                roi_norm_path = os.path.join(savedir, f"roi_norm{sl_block}.npy")
                if not os.path.exists(roi_norm_path):

                    st = offset - overlap
                    df_c = df[(df["Z"] >= sl_block * slice_num) & (df["Z"] < sl_block * slice_num + re)]
                    
                    reconst_pkl = os.path.join(pkldir, f"coordinates{sl_block}.pkl")
                    df_c.to_pickle(reconst_pkl)

                    img = read_tiff_stack(z_dir, start_index=st, num_images=re + overlap)
                    point_np = np.zeros((len(df_c), mask.size), dtype="float32")

                    for m in range(len(df_c)):
                        x, y, z = df_c.iloc[m]["X"] - 5, df_c.iloc[m]["Y"] - 5, df_c.iloc[m]["Z"] - 5
                        roi = img[int(z - st):int(z - st + 11), int(y):int(y + 11), int(x):int(x + 11)]
                        if roi.shape == mask.shape:
                            point_np[m] = np.ravel(roi)

                    del img
                    gc.collect()

                    roi_path = os.path.join(savedir, f"roi{sl_block}.npy")
                    if len(point_np) == 1:
                        np.save(roi_path, point_np)
                    elif len(point_np) == 0:
                        continue
                    else:
                        np.save(roi_path, point_np.T)

                    for m in range(len(point_np)):
                        point_np[m] = normalize(point_np[m])

                    np.save(roi_norm_path, point_np if len(point_np) == 1 else point_np.T)

                    del point_np
                    gc.collect()

                    # CUDA distance calculation
                    args = "{} {} {} {} {} {} {} {} {} {}".format(
                        os.path.join(dst, "cfos_roi", postfix) + os.sep, exps[l], dir, outf,
                        "psf", "mask", sl_block, size, blockdim_x, blockdim_y)
                    outc = f"./cal_d {args}"
                    print(outc)
                    subprocess.run([outc], shell=True)

/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT16_05/
          Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity  \
671              701  3113.0  4374.0   8.0    19060.0  18964.0      24.624683   
670              700  3841.0  4106.0   8.0     8163.0   7698.0      26.243809   
27012843           0  2871.0  4486.0  11.0    65535.0  65535.0     256.398132   

            min        ratioI  
671        96.0    198.541667  
670       465.0     17.554839  
27012843    0.0  65535.000000  
znum 2588
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT16_06/
      Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity  \
3783        3870   993.0  2791.0  10.0    16585.0  16373.0      18.038645   
3786        3873  2746.0  4648.0  13.0    26120.0  26039.0      24.607628   
3785        3872   770.0  4508.0  13.0    16476.0  16335.0      23.975323   

        min      ratioI  
3783  212.0   78.231132  
3786   81.0  322.469136  
3785

/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT28_04/
          Unnamed: 0       X       Y    Z  intensity   deltaI  dissimilarity  \
25224983           5   888.0  2579.0  7.0    65535.0  65530.0      34.384808   
63               119  3029.0  2523.0  8.0     6392.0   5831.0      28.400440   
64               120  3744.0  3643.0  8.0     5437.0   4828.0      30.849480   

            min        ratioI  
25224983    5.0  13107.000000  
63        561.0     11.393939  
64        609.0      8.927750  
znum 2468
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT28_05/
   Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity    min  \
0           0  1598.0  3043.0   8.0    12001.0  11688.0      22.240915  313.0   
1           1  2939.0  1001.0  10.0    24064.0  24056.0      33.295937    8.0   
2           2  3633.0   753.0  12.0    13673.0  13453.0      40.969173  220.0   

        ratioI  
0    38.341853  
1  3008.000000  
2    62.150000  
zn

/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT36_05/
    Unnamed: 0       X       Y    Z  intensity   deltaI  dissimilarity    min  \
32          38   796.0  2461.0  8.0    21985.0  21947.0      24.640730   38.0   
31          37  1801.0  1513.0  8.0     7007.0   6474.0      27.413441  533.0   
33          39   913.0  3126.0  8.0    41304.0  41177.0      17.839092  127.0   

        ratioI  
32  578.552632  
31   13.146341  
33  325.228346  
znum 2540
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT36_06/
     Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity  \
105         220  1138.0  2631.0  10.0     9340.0   8890.0      23.635468   
106         221   955.0  3343.0  10.0    11971.0  11689.0      25.940924   
107         222  1471.0  2806.0  13.0     6209.0   5639.0      29.153532   

       min     ratioI  
105  450.0  20.755556  
106  282.0  42.450355  
107  570.0  10.892982  
znum 2552
/home/gpu_data/data7/cfos_roi/output_pea

/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT40_06/
    Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity  \
36          48   583.0  2733.0   8.0     6723.0   6171.0      27.395601   
37          49  1220.0  3251.0  10.0     6316.0   5757.0      29.409382   
38          50  1518.0  4106.0  13.0    12606.0  12310.0      22.930548   

      min     ratioI  
36  552.0  12.179348  
37  559.0  11.298748  
38  296.0  42.587838  
znum 2520
6
Load tiff:1195-210
./cal_d /home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/ 1st cfos_CT40_06 dist psf mask 6 11 1 81
outf dist
blockdim_x  1
Using Device 0: NVIDIA RTX A6000
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT40_06/roi_norm6.npy
size:1437859 1331
i: 0 ,  roi_vs:0.163721
total points number  1437859
col_num  1331
/home/gpu_data/data7/cfos_roi/mask.npy
size:1331
i: 0 ,  mask[0]:0
mask_size,  1331
/home/gpu_data/data7/cfos_roi/psf.npy
size:81 1331
i: 0 ,  psf[0]:0.000001
psf_num,  8

size:1367495 1331
i: 0 ,  roi_vs:0.391843
total points number  1367495
col_num  1331
/home/gpu_data/data7/cfos_roi/mask.npy
size:1331
i: 0 ,  mask[0]:0
mask_size,  1331
/home/gpu_data/data7/cfos_roi/psf.npy
size:81 1331
./cal_d /home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/ 1st cfos_CT44_03 dist psf mask 6 11 1 81
outf dist
blockdim_x  1
Using Device 0: NVIDIA RTX A6000
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT44_03/roi_norm6.npy
size:1388652 1331
i: 0 ,  roi_vs:0.344251
total points number  1388652
col_num  1331
/home/gpu_data/data7/cfos_roi/mask.npy
size:1331
i: 0 ,  mask[0]:0
mask_size,  1331
/home/gpu_data/data7/cfos_roi/psf.npy
size:81 1331
i: 0 ,  psf[0]:0.000001
psf_num,  81
end vx_count
i: 0, j: 0,  min_dist:73.358879
i: 0, j: 1,  min_dist:93.329575
i: 0, j: 2,  min_dist:10.199966
i: 0, j: 3,  min_dist:110.993576
i: 0, j: 4,  min_dist:60.238514
i: 0, j: 5,  min_dist:53.103611
i: 0, j: 6,  min_dist:4.242227
i: 0, j: 7,  min_dist:33.412399
i: 0, 

/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT44_06/
       Unnamed: 0       X       Y     Z  intensity   deltaI  dissimilarity  \
57979       57992  1898.0  3466.0  10.0    21264.0  21106.0      23.404873   
57980       57993   160.0  4289.0  10.0    10777.0  10393.0      22.488628   
57981       57994  1088.0  2883.0  15.0     8100.0   7587.0      24.744663   

         min      ratioI  
57979  158.0  134.582278  
57980  384.0   28.065104  
57981  513.0   15.789474  
znum 2550
6
Load tiff:1195-210
7
Load tiff:1395-210
./cal_d /home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/ 1st cfos_CT44_06 dist psf mask 6 11 1 81
outf dist
blockdim_x  1
Using Device 0: NVIDIA RTX A6000
/home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/1st/cfos_CT44_06/roi_norm6.npy
./cal_d /home/gpu_data/data7/cfos_roi/output_peak_ratioI_fpr40/ 1st cfos_CT44_06 dist psf mask 7 11 1 81
outf dist
blockdim_x  1
Using Device 0: NVIDIA RTX A6000
/home/gpu_data/data7/cfos_roi/output_peak_ra

In [ ]:
# Quick result check: compute distance for first 3 ROIs

dir = "cfos_CT0_01"
size = 11

savedir = os.path.join(dst, "cfos_roi", postfix, exps[0], dir)
roi_path = os.path.join(savedir, "roi0.npy")
if not os.path.exists(roi_path):
    raise FileNotFoundError(f"ROI file not found: {roi_path}")

roi_np = np.load(roi_path).T
roi_np = roi_np.reshape(len(roi_np), size, size, size).astype("float32")

for i in range(min(3, len(roi_np))):
    print(GetDistance(roi_np[i], psfs, mask))

In [ ]:
# Load distance values from binary file

file = os.path.join(savedir, "dist_v.bin")
with open(file, "rb") as f:
    dist_v = np.fromfile(f, dtype=np.float32).astype("float32")

dist_v

In [ ]:
# Build a features dataframe from distance/tensor metrics

for l, exp in enumerate(exps):
    if l > 0:
        continue

    FP = os.path.join(src, deconv_fs[l])
    FPout = os.path.join(dst, f"{deconv_fs[l]}_{postfix}")

    # subdirectories under FPout
    dirs = [d for d in os.listdir(FPout) if os.path.isdir(os.path.join(FPout, d))]

    for n, dir in enumerate(dirs):
        feat_pkl = os.path.join(FPout, dir, "df_features.pkl")
        if os.path.exists(feat_pkl):
            continue

        print(exp, dir)
        z_dir = os.path.join(src, deconv_fs[l], dir)
        z_num = len(os.listdir(z_dir))
        print("znum", z_num)

        sl_block = z_num // slice_num
        re = z_num % slice_num

        # aggregate containers
        dist_np = None
        ratio_i_np = None
        sn_np = None
        tensor_np = None

        for k in range(sl_block):
            bin_path = os.path.join(dst, "cfos_roi", postfix, exp, dir, f"{outf}{k}.bin")
            with open(bin_path, "rb") as f:
                dist = np.fromfile(f, dtype=np.float32).reshape(-1, dist_num).astype("float32")

            roi_path = os.path.join(dst, "cfos_roi", postfix, exp, dir, f"roi{k}.npy")
            roi_np = np.load(roi_path).T
            roi_np = roi_np.reshape(-1, size * size * size).astype("float32")

            min_vals = np.min(roi_np, axis=1)
            max_vals = np.max(roi_np, axis=1)
            delta_i = max_vals - min_vals
            
            # Compute ratio only where min ≠ 0; otherwise set to 0
            ratio_i = np.zeros_like(max_vals, dtype="float32")
            nonzero = min_vals != 0
            ratio_i[nonzero] = max_vals[nonzero] / min_vals[nonzero]
            
            sn = delta_i
            tensor = calculate_inertia_tensor_diag_from_roi(roi_np, mask, size)

            if k == 0:
                dist_np = dist
                ratio_i_np = ratio_i
                sn_np = sn
                tensor_np = tensor
            else:
                dist_np = np.vstack([dist_np, dist])
                ratio_i_np = np.append(ratio_i_np, ratio_i)
                sn_np = np.append(sn_np, sn)
                tensor_np = np.append(tensor_np, tensor)

        if re != 0:
            bin_path = os.path.join(dst, "cfos_roi", postfix, exp, dir, f"{outf}{sl_block}.bin")
            if os.path.exists(bin_path):
                with open(bin_path, "rb") as f:
                    dist = np.fromfile(f, dtype=np.float32).reshape(-1, dist_num).astype("float32")

                roi_path = os.path.join(dst, "cfos_roi", postfix, exp, dir, f"roi{sl_block}.npy")
                roi_np = np.load(roi_path).T
                roi_np = roi_np.reshape(-1, size * size * size).astype("float32")

                min_vals = np.min(roi_np, axis=1)
                max_vals = np.max(roi_np, axis=1)
                delta_i = max_vals - min_vals
                
                # Compute ratio only where min ≠ 0; otherwise set to 0
                ratio_i = np.zeros_like(max_vals, dtype="float32")
                nonzero = min_vals != 0
                ratio_i[nonzero] = max_vals[nonzero] / min_vals[nonzero]
                
                sn = delta_i
                tensor = calculate_inertia_tensor_diag_from_roi(roi_np, mask, size)

                dist_np = np.vstack([dist_np, dist])
                ratio_i_np = np.append(ratio_i_np, ratio_i)
                sn_np = np.append(sn_np, sn)
                tensor_np = np.append(tensor_np, tensor)

        # assemble dataframe
        df = pd.DataFrame(np.hstack([dist_np, tensor_np.reshape(-1, 3)]))
        df.columns = [
            "Hellinger Coefficient", "Chernoff Coefficient", "Jeffreys Distance",
            "Directed Divergence", "J-Divergence", "L1", "L2", "Kullback-Leibler",
            "Pearson", "Jensen-Shannon", "Tensor_xx", "Tensor_yy", "Tensor_zz"
        ]
        df.insert(10, "delta I", sn_np)
        df.insert(10, "ratio I", ratio_i_np)

        df.replace(np.inf, 65535, inplace=True)
        print(df)

        out_pkl = os.path.join(FPout, dir, "df_features.pkl")
        df.to_pickle(out_pkl)
        print("save:", out_pkl)

In [ ]:
# Make teachers by ratioI threshold

dirs = ["cfos_CT0_01", "cfos_CT8_01", "cfos_CT16_01"]
FPw = os.path.join(dst, f"cfos_teachers/deconv_detection_ratioI_fpr{fpr}")
os.makedirs(FPw, exist_ok=True)

size = 11
center = size // 2

for i, acc_dir in enumerate(acc_dirs):
    for j, dir in enumerate(dirs):
        print(dir)

        teach_cord_dir = os.path.join(dst, acc_dir, "accuracy", dir)
        teach_cord_dir_ratio = os.path.join(dst, acc_dir, f"accuracy_ratioI_fpr{fpr}", dir)

        df_all_path = os.path.join(dst, deconv_fs[i] + f"_output_peak_ratioI_fpr{fpr}", dir, "coordinates.pkl")
        df_all = pd.read_pickle(df_all_path)  # all_peaks cut by ratioI threshold
        coords = df_all[['X', 'Y', 'Z']].values
        tree = KDTree(coords)

        for k, f in enumerate(os.listdir(teach_cord_dir)):
            if "th3750" in f and f.endswith(".csv"):
                name = f.split("_")[3]
                if name != "large":
                    continue
                print(name)

                df_cord = pd.read_csv(os.path.join(teach_cord_dir, f))
                print("all_deltaI_teacher:", len(df_cord))

                max_distance = 1
                dist, idx = tree.query(
                    np.column_stack([df_cord["X"].values, df_cord["Y"].values, df_cord["Z"].values]),
                    k=1, eps=0, p=2, distance_upper_bound=max_distance
                )

                idx = [ix for ix in idx if ix != len(df_all)]
                ex_ind = [n for n, ix in enumerate(idx) if ix != len(df_all)]

                df_cord = df_cord.iloc[ex_ind]
                print(f"teacher_ratioI_fpr{fpr}%: {len(df_cord)}")
                os.makedirs(teach_cord_dir_ratio, exist_ok=True)
                df_cord.to_csv(os.path.join(teach_cord_dir_ratio, f.replace("th3750", "ratioI")), index=False)

                st = int(np.min(df_cord["Z"]) - 10)
                print("slice start", st)

                num = 1015 if name == "large" else 65
                print("start", st)
                print("img num", num)

                img = read_tiff_stack(os.path.join(src, deconv_fs[i], dir), start_index=st, num_images=num)
                print(img.shape)

                out_raw_dir = os.path.join(FPw, exps[i], dir)
                os.makedirs(out_raw_dir, exist_ok=True)
                tifffile.imwrite(os.path.join(out_raw_dir, f"{name}_{st}_raw.tiff"), img)

                # positive
                peaks = []
                pn = "pos2"
                pos_npy_dir = os.path.join(out_raw_dir, f"teachingData_{pn}_npy")
                pos_tif_dir = os.path.join(out_raw_dir, f"teachingData_{pn}")
                os.makedirs(pos_npy_dir, exist_ok=True)
                os.makedirs(pos_tif_dir, exist_ok=True)

                df_pos = df_cord[(df_cord["Counter"] == 0) | (df_cord["Counter"] == 2)]
                for m in range(len(df_pos)):
                    x = int(df_pos.iloc[m]["X"]) - center
                    y = int(df_pos.iloc[m]["Y"]) - center
                    z = int(df_pos.iloc[m]["Z"]) - center
                    roi = img[z - (st - (size - center - 1)) : z - (st - (size - center - 1)) + size,
                              y : y + size,
                              x : x + size]
                    if roi.shape == mask.shape:
                        peaks.append(roi); peaks.append(roi)  # duplication
                        np.save(os.path.join(pos_npy_dir, f"{m}.npy"), roi)
                        tifffile.imwrite(os.path.join(
                            pos_tif_dir, f"{name}_{st}_{x}_{y}_{z}.tif"), roi)
                    else:
                        print("shape mismatch")

                # negative
                peaks = []
                pn = "neg2"
                neg_npy_dir = os.path.join(out_raw_dir, f"teachingData_{pn}_npy")
                neg_tif_dir = os.path.join(out_raw_dir, f"teachingData_{pn}")
                os.makedirs(neg_npy_dir, exist_ok=True)
                os.makedirs(neg_tif_dir, exist_ok=True)

                df_neg = df_cord[(df_cord["Counter"] == 1) | (df_cord["Counter"] == 3)]
                for n in range(len(df_neg)):
                    x = int(df_neg.iloc[n]["X"]) - center
                    y = int(df_neg.iloc[n]["Y"]) - center
                    z = int(df_neg.iloc[n]["Z"]) - center
                    roi = img[(z - st):(z - st + size), y:(y + size), x:(x + size)]
                    if roi.shape == mask.shape:
                        peaks.append(roi); peaks.append(roi)  # duplication
                        np.save(os.path.join(neg_npy_dir, f"{n}.npy"), roi)
                        tifffile.imwrite(os.path.join(
                            neg_tif_dir, f"{name}_{st}_{x}_{y}_{z}.tif"), roi)
                    else:
                        print("shape mismatch")

In [ ]:
# Make feature dataframe

tensor_pos = []
tensor_neg = []
tensor_psf = []
feature_pos = []
feature_neg = []

x_li = []
y_li = []
z_li = []

pname = "pos2"
nname = "neg2"
size = 11
center = size // 2

for l, exp in enumerate(exps):
    if l > 0:
        continue
    for n, dir in enumerate(dirs):
        pos_dir = os.path.join(FPw, exp, dir, f"teachingData_{pname}")
        neg_dir = os.path.join(FPw, exp, dir, f"teachingData_{nname}")

        pos_files = os.listdir(pos_dir)
        neg_files = os.listdir(neg_dir)

        print(len(pos_files))
        print(len(neg_files))

        # PSF tensors
        for f in psfs:
            tensor_psf.append(calculate_centroid_and_inertia_tensor_center_based(f * mask)[3])

        # Positive samples
        for f in pos_files:
            im = tifffile.imread(os.path.join(pos_dir, f))
            name = f.split("_")[0]
            x = int(f.split("_")[2]) + center
            y = int(f.split("_")[3]) + center
            z = int(f.split("_")[4].replace(".tif", "")) + center

            if im.shape == mask.shape:
                tensor_pos.append(calculate_centroid_and_inertia_tensor_center_based(im * mask)[3])
                tmp = GetDistance(normalize(im), psfs, mask)
                tmp.append(np.max(im) / (np.min(im) + 1))
                tmp.append((np.max(im) - np.min(im)) / np.min(im))
                tmp.append(exp)
                tmp.append(dir)
                tmp.append(name)
                tmp.append(x)
                tmp.append(y)
                tmp.append(z)
                feature_pos.append(tmp)

                x_li.append(x)
                y_li.append(y)
                z_li.append(z)

        # Negative samples
        for f in neg_files:
            im = tifffile.imread(os.path.join(neg_dir, f))
            name = f.split("_")[0]
            x = int(f.split("_")[2]) + center
            y = int(f.split("_")[3]) + center
            z = int(f.split("_")[4].replace(".tif", "")) + center

            if im.shape == mask.shape:
                tensor_neg.append(calculate_centroid_and_inertia_tensor_center_based(im * mask)[3])
                tmp = GetDistance(normalize(im), psfs, mask)
                tmp.append(np.max(im) / (np.min(im) + 1))
                tmp.append((np.max(im) - np.min(im)) / np.min(im))
                tmp.append(exp)
                tmp.append(dir)
                tmp.append(name)
                tmp.append(x)
                tmp.append(y)
                tmp.append(z)
                feature_neg.append(tmp)

In [ ]:
# Create feature dataframe and merge diagonal tensor components

df1 = create_dataframe(feature_pos, feature_neg)
df1.replace(np.inf, 65535, inplace=True)

tmp_p = [[t[0, 0], t[1, 1], t[2, 2]] for t in tensor_pos]
tmp_n = [[t[0, 0], t[1, 1], t[2, 2]] for t in tensor_neg]
tensor_df = pd.DataFrame(tmp_p + tmp_n, columns=["Tensor_xx", "Tensor_yy", "Tensor_zz"])

df2 = pd.concat([df1, tensor_df], axis=1)

meta_cols = ["exp", "CT", "region", "X", "Y", "Z"]
df_str = df2[meta_cols]
df_v = df2.drop(columns=meta_cols)
df2 = pd.concat([df_v, df_str], axis=1)
df2

In [ ]:
# Plot swarm distribution

plot_swarm(df2, features)

In [ ]:
# Plot pairwise feature relationships

Make2Dplots(df2, features)

In [ ]:
# Evaluate machine learning models

results = Machine_Learning(df2, features, test_size=0.3, random_state=42)

In [ ]:
# Hyperparameter tuning for XGBoost via Bayesian optimization, then save the best model

data = df2[features]
TF = df2["Group"].map({"pos": 1, "neg": 0})
data_train, data_test, TF_train, TF_test = train_test_split(
    data, TF, test_size=0.3, random_state=42
)

# Base model
model = xgb.XGBClassifier(objective="reg:squarederror", random_state=42)

# Search space
param_space = {
    "max_depth": Integer(1, 50),
    "learning_rate": Real(0.01, 1.0, prior="log-uniform"),
    "n_estimators": Integer(100, 3000),
    "min_child_weight": Integer(1, 10),
    "subsample": Real(0.1, 1.0),
    "colsample_bytree": Real(0.1, 1.0),
    "gamma": Real(1e-7, 0.5, prior="log-uniform"),
    "reg_alpha": Real(1e-7, 100, prior="log-uniform"),
    "reg_lambda": Real(1e-7, 100, prior="log-uniform"),
}

# Bayesian optimization
bayes_search = BayesSearchCV(
    estimator=model,
    search_spaces=param_space,
    n_iter=50,
    cv=5,
    n_jobs=20,
    verbose=2,
    scoring="f1",
    random_state=42,
)
bayes_search.fit(data_train, TF_train)

print("Best parameters found by Bayesian Optimization:")
print(bayes_search.best_params_)
print("Best cross-validation score: {:.4f}".format(-bayes_search.best_score_))

# Evaluate on the hold-out test set
best_model = bayes_search.best_estimator_
y_pred = best_model.predict(data_test)
f1 = f1_score(TF_test, y_pred)
print("F1 Score on test set: {:.4f}".format(f1))
print("\nClassification Report:")
print(classification_report(TF_test, y_pred))

# Feature importance plot
feature_importance = best_model.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5
fname_sorted = [features[i] for i in np.array(range(len(feature_importance)))[sorted_idx]]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(pos, feature_importance[sorted_idx], align="center")
ax.set_yticks(pos)
ax.set_yticklabels(fname_sorted)
ax.set_xlabel("Feature Importance")
ax.set_title("Feature Importance for XGBoost Model (Bayesian Optimization)")
plt.tight_layout()
fig_path = os.path.join(deconv_path_repre, f"Feature_Importance_XGBoost_fpr{fpr}.SVG")
fig.savefig(fig_path)
plt.show()

# Save tuned model
model_name = f"xgboost_best_ratioI_fpr{fpr}"
srcm = os.path.join(deconv_path_repre, "cd_model")
os.makedirs(srcm, exist_ok=True)
with open(os.path.join(srcm, model_name + ".pkl"), "wb") as file:
    pickle.dump(best_model, file)

In [ ]:
# Save best model using all features

model = "eXtreme Gradient Boosting (XGBoost)"

# Other available models for reference:
# "Logistic Regression": LogisticRegression(max_iter=1000)
# "Support Vector Machine (SVM)": SVC()
# "k-Nearest Neighbors (KNN)": KNeighborsClassifier()
# "Gradient Boosting": GradientBoostingClassifier()
# "Random Forest": RandomForestClassifier(n_estimators=100, random_state=random_state)
# "Decision Tree": DecisionTreeClassifier()
# "Multi-Layer Perceptron (MLP)": MLPClassifier(max_iter=1000)
# "Naive Bayes": GaussianNB()
# "eXtreme Gradient Boosting (XGBoost)": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
# "LightGBM": LGBMClassifier()
# "Adaptive Boosting (AdaBoost)": AdaBoostClassifier()

model_name = f"xgboost_ratioI_fpr{fpr}"
df_test_pred = Learn_save_best(df2, features, model, model_name, test_size=0.3, random_state=42)

print(df_test_pred)
len(df_test_pred)

In [ ]:
# Evaluate F1 score for each teacher region

region_names = ["SCN", "Striatum", "CA1", "SSpm23"]

pred = df_test_pred["predict"]

df2_pred = df2.iloc[df_test_pred.index.tolist()].copy()
df2_pred["predict"] = pred

print(df2_pred)

for region in region_names:
    df_r = df2_pred[df2_pred["region"] == region]
    labels = df_r["Group"].map({"pos": 1, "neg": 0}).tolist()

    f1 = f1_score(labels, df_r["predict"])
    print(f"f1 score, {region}: {f1:.3f}")

In [ ]:
# Prediction using trained model

model_src = os.path.join(src, "cd_model")
model_name = f"xgboost_best_ratioI_fpr{fpr}"

with open(os.path.join(model_src, model_name + ".pkl"), "rb") as file:
    model = pickle.load(file)

# Reconstruct peak coordinates and run prediction
for l, exp in enumerate(exps):
    if l > 0:
        continue

    FP = os.path.join(src, deconv_fs[l])
    FPout = os.path.join(dst, f"{deconv_fs[l]}_{postfix}")

    dirs = [d for d in os.listdir(FP) if os.path.isdir(os.path.join(FP, d))]

    for n, dir in enumerate(dirs):
        print(dir)

        feat_pkl = os.path.join(FPout, dir, "df_features.pkl")
        if not os.path.exists(feat_pkl):
            print(f"{feat_pkl}: not found")
            continue

        out_csv = os.path.join(FPout, dir, "coordinates_ai.csv")
        if os.path.exists(out_csv):
            continue

        df = pd.read_pickle(feat_pkl)

        z_dir = os.path.join(src, deconv_fs[l], dir)
        z_num = len(os.listdir(z_dir))
        print("znum", z_num)

        sl_block = z_num // slice_num
        re = z_num % slice_num

        # collect coordinate chunks
        reconst_dir = os.path.join(FPout, dir, "reconst")
        df_re_li = [pd.read_pickle(os.path.join(reconst_dir, f"coordinates{k}.pkl")) for k in range(sl_block)]
        if re != 0:
            df_re_li.append(pd.read_pickle(os.path.join(reconst_dir, f"coordinates{sl_block}.pkl")))
        df_re = pd.concat(df_re_li, axis=0)

        pred = model.predict(df)
        df_re["predict"] = pred

        df_re.to_csv(out_csv, index=False)
        print("save:", out_csv)
        print("positive:", np.sum(pred == 1))
        print("negative:", np.sum(pred == 0))
        print("pred_df", df_re)

znum 2478
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT0_06/coordinates_ai.csv
positive: 1775609
negative: 9189829
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
49                64  1893.0  3698.0     8.0     6367.0   5815.0   
50                65  1382.0   218.0    12.0    21092.0  21043.0   
52                67   210.0  3191.0    13.0    13386.0  13122.0   
51                66  1345.0  3007.0    13.0     9791.0   9457.0   
53                68  2628.0  2763.0    15.0     5284.0   4647.0   
...              ...     ...     ...     ...        ...      ...   
22506835    22513472  2078.0  3648.0  2457.0     6704.0   6155.0   
22506836    22513473  3666.0  2853.0  2466.0     9192.0   8790.0   
22506847    22513485  1150.0  2658.0  2467.0     7897.0   7398.0   
22506848    22513486  2914.0  4384.0  2469.0     9525.0   9153.0   
22508149        1300  3423.0  3207.0  2471.0 

znum 2538
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT12_05/coordinates_ai.csv
positive: 806877
negative: 14503630
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
589              589  3091.0  1323.0     8.0    12621.0  12327.0   
26478043           0  2789.0  3882.0     9.0    65535.0  65535.0   
591              605  2461.0  3758.0    10.0    12978.0  12703.0   
590              604  1203.0  3541.0    10.0    16777.0  16520.0   
592              615  3956.0  3886.0    10.0    10993.0  10635.0   
...              ...     ...     ...     ...        ...      ...   
26478035    26485931  3599.0  3583.0  2514.0    12092.0  11792.0   
26478036    26485932  2581.0  3606.0  2514.0     8750.0   8277.0   
26478040    26485936  3904.0  4154.0  2517.0     8636.0   8176.0   
26478041    26485937   884.0  4117.0  2524.0    20434.0  20374.0   
26478042    26485938  3676.0  2389.0  2531.0

znum 2620
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT16_04/coordinates_ai.csv
positive: 975500
negative: 14604324
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
394              394   530.0  4194.0     9.0    19960.0  19887.0   
395              397    65.0  4174.0    10.0     9957.0   9544.0   
27173636           0  1891.0  3593.0    10.0    65535.0  65535.0   
396              398  2709.0  3974.0    11.0    11125.0  10850.0   
397              399  2493.0  3773.0    12.0    16695.0  16563.0   
...              ...     ...     ...     ...        ...      ...   
27173626    27185525  3869.0  2813.0  2611.0     9057.0   8586.0   
27173623    27185522  3796.0  2641.0  2611.0     7876.0   7383.0   
27173632    27185531  3536.0  1983.0  2611.0    12944.0  12640.0   
27175242        1606   987.0  2463.0  2611.0    65535.0  65535.0   
27173627    27185526  1241.0   730.0  2613.0

znum 2610
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT20_03/coordinates_ai.csv
positive: 1084136
negative: 9658368
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
547              657   975.0  3698.0     8.0     6358.0   5811.0   
546              656  3749.0  3228.0     8.0     9826.0   9422.0   
544              654  2882.0  1075.0     8.0     9703.0   9375.0   
545              655  2336.0  2156.0     8.0     5725.0   5132.0   
548              658  3401.0  3946.0    13.0    11329.0  10991.0   
...              ...     ...     ...     ...        ...      ...   
24715534    24721670  3403.0  1928.0  2594.0    19088.0  18970.0   
24715535    24721671  1701.0  3588.0  2594.0     7409.0   6903.0   
24715536    24721672  1698.0  4636.0  2595.0     8499.0   8282.0   
24715537    24721673  1698.0  4647.0  2600.0    15229.0  15145.0   
24715538    24721674  3391.0  1560.0  2601.0

znum 2460
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT24_02/coordinates_ai.csv
positive: 2300946
negative: 9913561
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
6330            6330  3436.0  1936.0     7.0    17043.0  16943.0   
6331            6331   128.0  2671.0     7.0     7896.0   7511.0   
6332            6332  1391.0  1208.0     8.0     7267.0   6767.0   
6333            6333   730.0  3981.0     8.0    11534.0  11199.0   
6334            6334  3339.0  1673.0    10.0     6119.0   5546.0   
...              ...     ...     ...     ...        ...      ...   
22815883    22832645  2648.0  3041.0  2439.0    10608.0  10243.0   
22815884    22832646  1144.0  2850.0  2442.0    28026.0  27956.0   
22815863    22832572  2151.0  2228.0  2444.0     5614.0   4972.0   
22815864    22832573  3861.0  1347.0  2446.0    10962.0  10654.0   
22815865    22832574  3079.0  1235.0  2448.0

...

znum 2568
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT40_03/coordinates_ai.csv
positive: 1607312
negative: 10133941
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
24                37  2546.0  3381.0    10.0     7343.0   6816.0   
16                29  2368.0  5179.0    13.0    10909.0  10726.0   
25                38  2638.0  3423.0    13.0     6700.0   6168.0   
17                30  3101.0  3108.0    18.0     9765.0   9361.0   
18                31   505.0  3293.0    21.0     9442.0   9049.0   
...              ...     ...     ...     ...        ...      ...   
23996235    24005515   443.0  3100.0  2551.0    32676.0  32676.0   
23996236    24005516   688.0  2191.0  2554.0     5862.0   5262.0   
23996252    24005532   165.0  2056.0  2556.0     7020.0   6513.0   
23996250    24005530  1426.0  1195.0  2556.0     5472.0   4880.0   
23996251    24005531  3191.0  1668.0  2556.

znum 2535
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT44_02/coordinates_ai.csv
positive: 1224156
negative: 9854926
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
0                  0  1608.0   802.0     8.0    19460.0  19356.0   
1                  1  2171.0  2793.0    15.0     8085.0   7568.0   
2                  2   818.0  1978.0    16.0    38901.0  38901.0   
3                  3  3765.0  1028.0    20.0     9468.0   9048.0   
4                  4  2799.0  1423.0    21.0    17551.0  17444.0   
...              ...     ...     ...     ...        ...      ...   
24817752    24825146   410.0  3321.0  2519.0     5774.0   5131.0   
24817758    24825152  3696.0  1760.0  2520.0     7185.0   6706.0   
24817755    24825149  1240.0  3113.0  2520.0     8698.0   8303.0   
24817756    24825150  2875.0  3070.0  2521.0    16664.0  16497.0   
24817754    24825148  3461.0  3118.0  2524.0

znum 2408
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT4_01/coordinates_ai.csv
positive: 685324
negative: 9494767
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
330984        331078  1453.0  2992.0     7.0    18265.0  18213.0   
330985        331079  1448.0  3005.0     8.0    17774.0  17768.0   
330986        331080  1075.0  2475.0    11.0    26361.0  26361.0   
330988        331082   996.0  5516.0    12.0    14600.0  14589.0   
330987        331081   994.0  5511.0    12.0    15207.0  15196.0   
...              ...     ...     ...     ...        ...      ...   
24722334        1700  2469.0  3069.0  2386.0    65535.0  65535.0   
24720630    24732550  1390.0  3787.0  2390.0    11216.0  10937.0   
24720631    24732551  2136.0  3798.0  2394.0     5447.0   4819.0   
24720633    24732553  1578.0  1448.0  2399.0    18200.0  17997.0   
24720632    24732552  2721.0  1333.0  2399.0  

znum 2485
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT4_06/coordinates_ai.csv
positive: 1022437
negative: 10158943
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
16378          16378  3386.0  4354.0     7.0     7864.0   7462.0   
24364383           0   963.0  4539.0     8.0    65535.0  65535.0   
16379          16434  2489.0  5713.0     9.0    29271.0  29271.0   
16380          16435  1298.0   330.0    10.0     6254.0   5684.0   
16381          16436  3631.0  2651.0    10.0     6444.0   5890.0   
...              ...     ...     ...     ...        ...      ...   
24364381    24428772  2121.0  2108.0  2461.0     5600.0   4960.0   
24364382    24428773  3666.0  3832.0  2461.0    40510.0  40510.0   
24364359    24428747  2866.0  1170.0  2465.0     9935.0   9535.0   
24364360    24428748  1084.0  3733.0  2466.0     8029.0   7598.0   
24364361    24428749  2551.0  3135.0  2471.0

znum 2542
save: /home/gpu_data/data7//230828circadian_Data1/230828circadian_1st_Reconst_median_norm_Deconv_output_peak_ratioI_fpr40/cfos_CT8_05/coordinates_ai.csv
positive: 571065
negative: 16235930
pred_df           Unnamed: 0       X       Y       Z  intensity   deltaI  \
27925271           7  1487.0  4093.0     7.0    65535.0  65535.0   
221429        221543  1170.0  3398.0     8.0     8013.0   7551.0   
221430        221544  2656.0  4011.0     8.0    11751.0  11419.0   
221431        221545  2506.0  3591.0     9.0    20673.0  20513.0   
221432        221546  1254.0  3710.0    13.0    55429.0  55429.0   
...              ...     ...     ...     ...        ...      ...   
27925259    27935380  3364.0  3894.0  2532.0    14505.0  14316.0   
27925262    27935383   653.0  2641.0  2533.0    10078.0   9689.0   
27925261    27935382  3234.0  1533.0  2533.0    15326.0  15134.0   
27925260    27935381   972.0  1475.0  2533.0    26019.0  26019.0   
27925263    27935384  2546.0  2771.0  2533.0 